## 基本环境 · Basic setup

首次打开运行下面 3 个 cell。它们做的事:
1. 把工作目录切到 `solutions/` (这样 `from attention.mha import ...` 这种导入能直接生效)。
2. 启用 `autoreload`，编辑 .py 文件保存后 notebook 里立刻可用，不用重启 kernel。
3. 设 `LAYERNORM_TYPE=torch`，避免 CUDA-only 算子的 import 失败。

First time you open the notebook, run the 3 cells below: cd into `solutions/`, turn on autoreload, force the pure-PyTorch LayerNorm path.

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) != 'solutions':
    if os.path.isdir('solutions'):
        os.chdir('solutions')
    else:
        # already inside a chapter folder — climb out
        while os.path.basename(os.getcwd()) != 'solutions' and os.getcwd() != '/':
            os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
os.environ.setdefault('LAYERNORM_TYPE', 'torch')
print('cwd =', os.getcwd())

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
# Folder where the attention chapter's reference .pt files live
control_folder = 'attention/control_values'
assert os.path.isdir(control_folder), f'missing {control_folder}'

# 第 1 章 · Attention

AF3 的所有 attention 流派 (token / atom / pair-biased / 局部窗口)，底层都用同一个多头注意力 + 一组初始化好的线性层。本章把这些底座的代码一行一行填出来。

本章涉及的模块 (全部位于 `attention/`):

| 文件 | 类 / 函数 | 作用 |
|---|---|---|
| `linear.py` | `Linear._init_params`, `Linear.forward`, `BiasInitLinear.__init__` | 自定义初始化的线性层 |
| `layer_norm.py` | `OpenFoldLayerNorm` | 与 Protenix 融合算子参数名对齐的 LN |
| `mha.py` | `_attention`, `Attention._prep_qkv`, `Attention._wrap_up`, `Attention.__init__`, `Attention.forward` | 多头注意力 |
| `transition.py` | `AdaptiveLayerNorm`, `Transition` | AF3 算法 26 / 11 |
| `attention_pair_bias.py` | `AttentionPairBias` | AF3 算法 24 |

每个测试在调用前会把模块参数替换成 `linspace(-1, 1, numel)`，只要你的实现与参考相同，输出就会完全相同 (`torch.allclose`)。

## 1.1 Linear

打开 `attention/linear.py`，把 `Linear._init_params`、`Linear.forward` 和`BiasInitLinear.__init__` 三个 TODO 填好。

Linear 在 AF3 不同位置初始化方式不同 (`default` / `relu` / `zeros`)，Linear.forward 的 precision 路径用于强制 fp32 计算 (坐标投影、噪声水平条件)。BiasInitLinear 是为 sigmoid 门设定初始开度的特殊 Linear。

In [ ]:
from attention.linear import Linear, LinearNoBias, BiasInitLinear
from attention.control_values.attention_checks import (
    c_a, c_s, c_z, test_module_shape, test_module_forward,
    test_inputs,
)

# Linear with default initializer
lin = Linear(in_features=c_a, out_features=c_z)
test_module_shape(lin, 'linear_default', control_folder)
test_module_forward(lin, 'linear_default',
                    inputs=(test_inputs['x_a'],),
                    output_names='out',
                    control_folder=control_folder)

# LinearNoBias
lin_nb = LinearNoBias(in_features=c_a, out_features=c_z)
test_module_shape(lin_nb, 'linear_nobias', control_folder)
test_module_forward(lin_nb, 'linear_nobias',
                    inputs=(test_inputs['x_a'],),
                    output_names='out',
                    control_folder=control_folder)

# BiasInitLinear with biasinit=-2.0
bil = BiasInitLinear(in_features=c_s, out_features=c_a, bias=True, biasinit=-2.0)
test_module_shape(bil, 'bias_init_linear', control_folder)
test_module_forward(bil, 'bias_init_linear',
                    inputs=(test_inputs['x_s'],),
                    output_names='out',
                    control_folder=control_folder)
print('Linear ✓')

## 1.2 LayerNorm

打开 `attention/layer_norm.py`，把 `OpenFoldLayerNorm.__init__` 和`OpenFoldLayerNorm.forward` 两个 TODO 填好。AdaLN 等需要`create_scale=False` 或 `create_offset=False`，因此 LN 必须支持可选的 scale/offset。

In [ ]:
from pairformer.triangle_ops import LayerNorm

ln = LayerNorm(c_a)
test_module_shape(ln, 'layer_norm', control_folder)
test_module_forward(ln, 'layer_norm',
                    inputs=(test_inputs['x_a'],),
                    output_names='out',
                    control_folder=control_folder)
print('LayerNorm ✓')

## 1.3 `_attention` (核心点积数学)

打开 `attention/mha.py`，把模块顶部的 `_attention(q, k, v, attn_bias, ...)` 函数 TODO 填好。

这是所有 attention 流派最底层的数学：缩放点积 + softmax + 加权求和。函数没有可学参数，所以测试直接比较输出张量。

In [ ]:
from attention.mha import _attention

out = _attention(
    test_inputs['q_raw'].double(),
    test_inputs['k_raw'].double(),
    test_inputs['v_raw'].double(),
    attn_bias=None,
    use_efficient_implementation=False,
)
expected = torch.load(f'{control_folder}/attention_function_out.pt')
assert torch.allclose(out, expected), '_attention output mismatch'
print('_attention ✓')

## 1.4 Attention (多头注意力模块)

继续在 `mha.py` 里，依次填:
- `Attention.__init__` (5 个线性层 + 可选门控)
- `Attention._prep_qkv` (Q/K/V 投影 + 拆头 + 缩放)
- `Attention._wrap_up` (可选 sigmoid 门控 + 展平 + 输出投影)
- `Attention.forward` (调度局部 / 全连接两条路径)

In [ ]:
from attention.mha import Attention
from attention.control_values.attention_checks import N_head, c_hidden

attn = Attention(
    c_q=c_a, c_k=c_a, c_v=c_a,
    c_hidden=c_hidden, num_heads=N_head,
    gating=True, q_linear_bias=True,
    use_efficient_implementation=False,
    zero_init=False,
)
test_module_shape(attn, 'mha_gated', control_folder)

from attention.control_values.attention_checks import test_module_method
test_module_method(
    attn, 'mha_gated',
    inputs=(test_inputs['q_x'], test_inputs['kv_x'], test_inputs['attn_bias']),
    output_names='out',
    control_folder=control_folder,
    method=lambda q_x, kv_x, b: attn(q_x=q_x, kv_x=kv_x, attn_bias=b),
)
print('Attention ✓')

## 1.5 AdaptiveLayerNorm + Transition

打开 `attention/transition.py`，填两个 TODO:
- `AdaptiveLayerNorm.__init__` 和 `.forward` (AF3 算法 26，FiLM 风格调制)
- `Transition.__init__` 和 `.forward` (AF3 算法 11，SwiGLU FFN)

In [ ]:
from attention.transition import AdaptiveLayerNorm, Transition
from attention.control_values.attention_checks import n_factor

adaln = AdaptiveLayerNorm(c_a=c_a, c_s=c_s)
test_module_shape(adaln, 'adaptive_layer_norm', control_folder)
test_module_forward(adaln, 'adaptive_layer_norm',
                    inputs=(test_inputs['x_a'], test_inputs['x_s']),
                    output_names='out',
                    control_folder=control_folder)

tr = Transition(c_in=c_a, n=n_factor)
test_module_shape(tr, 'transition', control_folder)
test_module_forward(tr, 'transition',
                    inputs=(test_inputs['x_a'],),
                    output_names='out',
                    control_folder=control_folder)
print('AdaLN + Transition ✓')

## 1.6 AttentionPairBias (AF3 算法 24)

打开 `attention/attention_pair_bias.py`，把 `__init__`、`local_multihead_attention`、`standard_multihead_attention`、`forward` 四个 TODO 填好。

AttentionPairBias 是 AF3 最核心的复合块: AdaLN + 多头 attention + pair bias + adaLN-Zero 输出门。

In [ ]:
from attention.attention_pair_bias import AttentionPairBias

apb = AttentionPairBias(
    has_s=True, create_offset_ln_z=False,
    n_heads=N_head, c_a=c_a, c_s=c_s, c_z=c_z,
    biasinit=-2.0, cross_attention_mode=False,
)
# Disable the SDP fast path so the test runs in double precision.
apb.attention.use_efficient_implementation = False

test_module_shape(apb, 'attention_pair_bias', control_folder)
test_module_method(
    apb, 'attention_pair_bias',
    inputs=(test_inputs['x_a'], test_inputs['x_s'], test_inputs['x_z']),
    output_names='out',
    control_folder=control_folder,
    method=lambda a, s, z: apb(a=a, s=s, z=z),
)
print('AttentionPairBias ✓')

## 章节小结

完成本章后，你已经实现了:
- 三种自定义初始化策略的 Linear (default / relu / zeros) 与 sigmoid 门 BiasInitLinear
- 与 Protenix 算子参数名对齐的 LayerNorm
- 多头注意力的完整链路 (`_attention` 数学 + `_prep_qkv` 形状处理 + `_wrap_up` 门控)
- AF3 三个高频组件: `AdaptiveLayerNorm` / `Transition` / `AttentionPairBias`

这些是后续章节 Pairformer / Diffusion 的基石。